# Synthetic Multifactor Market 50D EDA

This output-stripped notebook is a guarded exploratory analysis for the experimental 50-dimensional synthetic multifactor benchmark. It generates only a small in-memory smoke batch by default, reads experimental profile metadata, and prints non-smoke commands without running them.

The benchmark note is `docs/benchmarks/multifactor_market.md`. No checkpoints, generated output files, or registry updates are produced by this notebook.

## Benchmark formulation and tensor convention

The no-jump synthetic return model is

$$r_t = B f_t + \epsilon_t,$$

where $r_t \in \mathbb{R}^{50}$ is a log-return vector, $B$ is a sector-structured loading matrix, $f_t \in \mathbb{R}^{5}$ is a stochastic-volatility factor return vector, and $\epsilon_t$ is idiosyncratic asset noise. The jump stress profile adds

$$r_t = B f_t + \epsilon_t + J_t, \qquad J_t = J_t^{common} + J_t^{sector}.$$

The standard tensor convention is `data: [n_sample, 60, 50]` and `labels: [n_sample, 1]` under `condition_mode: constant`. Loadings, sectors, regimes, covariance summaries, and jump masks are oracle diagnostics, not model-visible inputs.

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

RUN_TRAINING = False
RUN_EVALUATION = False
RUN_FULL = False
ALLOW_MISSING_OUTPUTS = True

REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT.parent != REPO_ROOT and not (REPO_ROOT / "pyproject.toml").exists():
    REPO_ROOT = REPO_ROOT.parent

if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib-time-causal-vae")
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display

from time_causal_vae.data.multifactor_market import MultifactorMarketDataset
from time_causal_vae.experiments.multidim_profiles import select_multidim_profile

torch.manual_seed(0)
np.random.seed(0)

N_SAMPLES = 512 if RUN_FULL else 96
N_TIMESTEPS = 60
N_ASSETS = 50
N_FACTORS = 5
N_SECTORS = 5

## Generate a smoke batch in memory

This cell uses the train-style no-jump benchmark settings: fixed market structure, independent path seed, 60 time steps, 50 assets, 5 factors, 5 sectors, and train-style per-asset return standardisation. It does not write any artefacts.

In [ ]:
dataset = MultifactorMarketDataset(
    N_SAMPLES,
    N_TIMESTEPS,
    n_assets=N_ASSETS,
    n_factors=N_FACTORS,
    n_sectors=N_SECTORS,
    structure_seed=0,
    path_seed=0,
    condition_mode="constant",
    with_jumps=False,
    standardize_returns=True,
)

raw_returns = dataset.raw_log_returns.detach().cpu()
model_visible = dataset.data.detach().cpu()
labels = dataset.labels.detach().cpu()
loadings = dataset.loadings.detach().cpu()
sector_labels = dataset.sector_labels.detach().cpu()

summary = pd.DataFrame([
    {
        "tensor": "raw_returns",
        "shape": tuple(raw_returns.shape),
        "finite": bool(torch.isfinite(raw_returns).all()),
    },
    {
        "tensor": "model_visible",
        "shape": tuple(model_visible.shape),
        "finite": bool(torch.isfinite(model_visible).all()),
    },
    {
        "tensor": "labels",
        "shape": tuple(labels.shape),
        "finite": bool(torch.isfinite(labels).all()),
    },
    {
        "tensor": "loadings",
        "shape": tuple(loadings.shape),
        "finite": bool(torch.isfinite(loadings).all()),
    },
])
display(summary)
display(
    Markdown(
        f"Standardisation metadata source: `{dataset.metadata['standardization']['stats_source']}`."
    )
)

## Sector labels and loading visualisation

Sector labels and loadings are simulator metadata. They are useful for EDA and diagnostics, but they are not model-visible conditions in the benchmark.

In [ ]:
sector_frame = pd.DataFrame({
    "asset_index": np.arange(N_ASSETS),
    "sector_id": sector_labels.numpy(),
})
display(sector_frame.groupby("sector_id").size().rename("asset_count").reset_index())

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), constrained_layout=True)
im = axes[0].imshow(loadings.numpy(), aspect="auto", cmap="coolwarm")
axes[0].set_title("Factor loadings")
axes[0].set_xlabel("Factor")
axes[0].set_ylabel("Asset")
fig.colorbar(im, ax=axes[0], fraction=0.046, pad=0.04)
axes[1].step(np.arange(N_ASSETS), sector_labels.numpy(), where="mid")
axes[1].set_title("Sector label by asset")
axes[1].set_xlabel("Asset")
axes[1].set_ylabel("Sector")
display(fig)
plt.close(fig)

## Covariance, correlation, and eigenspectrum diagnostics

These are projection-free smoke diagnostics on the raw 50D returns. Non-smoke reports should compare generated paths against held-out reference paths using the full cross-sectional diagnostic suite.

In [ ]:
flat_returns = raw_returns.reshape(-1, N_ASSETS).numpy()
cov = np.cov(flat_returns, rowvar=False)
std = np.sqrt(np.clip(np.diag(cov), a_min=1e-12, a_max=None))
corr = cov / np.outer(std, std)
eigvals = np.linalg.eigvalsh(corr)[::-1]

sector_ids = sector_labels.numpy()
sector_rows = []
for sector_id in sorted(set(sector_ids.tolist())):
    mask = sector_ids == sector_id
    block = corr[np.ix_(mask, mask)]
    off_diag = block[~np.eye(block.shape[0], dtype=bool)]
    sector_rows.append({
        "sector_id": int(sector_id),
        "asset_count": int(mask.sum()),
        "mean_off_diagonal_corr": float(off_diag.mean()) if off_diag.size else np.nan,
    })

diagnostic_summary = pd.DataFrame([
    {"metric": "mean_asset_volatility", "value": float(std.mean())},
    {"metric": "max_asset_volatility", "value": float(std.max())},
    {"metric": "top_corr_eigenvalue", "value": float(eigvals[0])},
    {"metric": "top5_corr_eigenvalue_mass", "value": float(eigvals[:5].sum() / eigvals.sum())},
])
display(diagnostic_summary)
display(pd.DataFrame(sector_rows))

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), constrained_layout=True)
corr_image = axes[0].imshow(corr, vmin=-1, vmax=1, cmap="coolwarm")
axes[0].set_title("Smoke-batch correlation")
fig.colorbar(corr_image, ax=axes[0], fraction=0.046, pad=0.04)
axes[1].plot(np.arange(1, 16), eigvals[:15], marker="o")
axes[1].set_title("Top correlation eigenvalues")
axes[1].set_xlabel("Rank")
axes[1].set_ylabel("Eigenvalue")
display(fig)
plt.close(fig)

## Experimental profile metadata

Multidimensional model profiles are metadata for local comparisons. This notebook reads `trained_models/multidim_profiles.yaml`, not `trained_models/model_registry.yaml`; no multidimensional public default exists.

In [ ]:
profile_names = ["correlation_sector", "portfolio_tail"]
profile_rows = []
for profile_name in profile_names:
    selection = select_multidim_profile("multifactor_market", profile_name).to_dict()
    metadata = selection["metadata"]
    profile_rows.append({
        "profile": profile_name,
        "family": selection["family"],
        "candidate": metadata["candidate"],
        "public_default": selection["public_default"],
        "summary": metadata["evidence"]["profile_summary"],
        "caveat_count": len(metadata.get("caveats", [])),
    })
display(pd.DataFrame(profile_rows))

rvq_profile = select_multidim_profile("multifactor_market", "portfolio_tail").metadata
display(
    pd.DataFrame([
        {
            "tokenizer_config": rvq_profile.get("tokenizer_config"),
            "prior_config": rvq_profile.get("prior_config"),
            "temperature": rvq_profile.get("sampling", {}).get("temperature"),
            "top_k": rvq_profile.get("sampling", {}).get("top_k"),
        }
    ])
)

## Non-smoke commands to run manually

The following commands are printed for convenience. This notebook does not execute them.

In [ ]:
commands = [
    "poetry run python scripts/run_multifactor_continuous_seed_robustness.py",
    "poetry run python scripts/evaluate_multifactor_rvq_token_prior.py --config configs/experiments/multifactor_market_factor_pca_rvq_q2_cb64_prior_factorised_additive.yaml",
    "poetry run python scripts/run_multifactor_sampling_calibration.py --help",
]

print("Notebook guards:")
print(f"RUN_TRAINING={RUN_TRAINING}")
print(f"RUN_EVALUATION={RUN_EVALUATION}")
print(f"RUN_FULL={RUN_FULL}")
print()
for command in commands:
    print(command)

if RUN_TRAINING or RUN_EVALUATION:
    display(
        Markdown(
            "Training/evaluation guards are true, but this EDA notebook still prints commands only."
        )
    )